# Tracking y odometría visual

1. En Colab, selecciona GPU si está disponible.
2. Elige una experiencia y ejecuta las celdas en orden.
3. Abre el **LINK PÚBLICO** para usar la aplicación.
4. Descarga los resultados antes de detener la celda de ejecución.

El notebook descarga automáticamente el código de `apps/` desde la rama `main` de GitHub. No necesitas copiar archivos ni actualizar el código dentro del notebook.

Para cambiar de experiencia, detén la ejecución anterior, cambia el selector y repite las celdas. Los archivos del entorno son temporales. Quien tenga el enlace público puede acceder a la aplicación.

## Etapa 01 · Conceptos

Tracking sigue objetos entre imágenes; la odometría visual estima el movimiento de la cámara. Se complementan para distinguir el movimiento de los objetos del movimiento de quien observa.

Preguntas que uno debe poder responder al finalizar la actividad:

- Qué es la odometría visual, sus pasos y tipos.
- Ventajas y desventajas frente a la odometría de ruedas.
- Relación con VIO y SLAM.
- Al menos dos aplicaciones en visión artificial.
- Qué es tracking, sus diferencias y su relación con la odometría.
- Su uso en realidad aumentada, experiencias inmersivas y VFX/cine.

## Etapa 02 · Aplicaciones y conceptos

Selecciona **Etapa 02 - Tracking 2D**. La interfaz contiene los cuatro apartados:

1. **a · Comparación:** explora las animaciones de tracking 2D frente a estimación 3D.
2. **b · Tracking:** elige video o webcam fija, ByteTrack o BoT-SORT y una línea de conteo. Observa IDs y trayectorias.
3. **c · Velocidad y zonas:** consulta px/s, cuadrícula y mapa de calor. Para estimar m/s, pulsa «Marcar referencia», marca dos puntos, introduce los metros y aplica. En video, calibra antes de analizar.
4. **d · Cámara móvil:** abre la cámara ante un fondo plano con textura. Cuando esté listo, captura la referencia, mueve suavemente la cámara y compara trayectorias originales y compensadas.

Descarga los CSV antes de detener la sesión. El mapa acumula detecciones. La escala métrica es local y depende de la perspectiva. La compensación con ORB trabaja sobre un plano en píxeles de referencia; no es odometría 3D.

## Etapa 03 · Monitor Inteligente de Tráfico y Navegación

| Nivel | Selección | Qué hacer |
| --- | --- | --- |
| 1 · Radar con cámara fija | Niveles 1 y 2: Radar y analítica | Carga un MP4, marca P1/P2 sobre el suelo e introduce su distancia real. Puedes medirla manualmente en Google Maps. Configura el límite y observa km/h y alertas. |
| 2 · Analítica de tránsito | La misma ejecución del radar | Descarga CSV, mapa de calor y gráfica. Indica la hora inicial de grabación para agrupar automóviles por hora. |
| 3 · Cámara en movimiento | Nivel 3: Odometría visual | Usa una webcam móvil o video caminando. Ajusta altura y campo de visión. Compara velocidades aparentes y compensadas y observa la trayectoria estimada con ORB. |

**Comparación:** prueba primero un video móvil en el radar y después el mismo video en odometría. Observa un auto estacionado y registra si la compensación reduce su velocidad aparente.

**Límites:** el radar admite MP4 y genera reportes al finalizar el análisis. Su CSV resume cada ID y el mapa muestra zonas, sin clasificar carriles. La perspectiva afecta la escala; la odometría depende del suelo y la calibración y puede acumular error. La velocidad compensada es una estimación, no una medición de referencia.

In [ ]:
# @title 1. Elegir experiencia
from pathlib import Path
import os
import platform
import shutil
import signal
import socket
import subprocess
import sys
import tempfile

experiencia = "Etapa 02 - Tracking 2D" # @param ["Etapa 02 - Tracking 2D", "Etapa 03 - Niveles 1 y 2: Radar y analítica", "Etapa 03 - Nivel 3: Odometría visual"]
APPLICATIONS = {
    "Etapa 02 - Tracking 2D": "tracking_2d",
    "Etapa 03 - Niveles 1 y 2: Radar y analítica": "radar_trafico",
    "Etapa 03 - Nivel 3: Odometría visual": "odometria_visual",
}
if platform.system() != "Linux" or platform.machine() not in ("x86_64", "AMD64"):
    raise RuntimeError("Este notebook prepara un túnel Linux x86_64. Utiliza Google Colab.")
if experiencia not in APPLICATIONS:
    raise ValueError("Selecciona una experiencia de la lista.")
REPOSITORY = "https://github.com/elisbanpaco/tracking-y-odometria-visual.git"
APP = None
INSTALLED_APP = None
print(experiencia)
print("Python:", sys.version.split()[0])

In [ ]:
# @title 2. Descargar el código desde GitHub
APP = None
INSTALLED_APP = None
if not shutil.which("git"):
    raise RuntimeError("No se encontró Git. Utiliza un entorno de Google Colab.")
ROOT = Path(tempfile.mkdtemp(prefix="tracking-odometria-")) / "repositorio"
try:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(ROOT)], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError("No se pudo descargar el proyecto. Comprueba la conexión y el acceso al repositorio de GitHub, y repite esta celda.") from exc
selected = ROOT / "apps" / APPLICATIONS[experiencia]
for filename in ("app.py", "requirements.txt"):
    if not (selected / filename).is_file():
        raise RuntimeError(f"Falta {filename} en {selected.name}. Publica la estructura apps/ en la rama main antes de continuar.")
revision = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
APP = selected
print("Experiencia:", experiencia)
print("Revisión descargada:", revision)


In [ ]:
# @title 3. Instalar con uv y preparar la experiencia
INSTALLED_APP = None
if APP is None:
    raise RuntimeError("Descarga primero el código del proyecto en la celda anterior.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--system", "--python", sys.executable, "-r", str(APP / "requirements.txt")], check=True)
needed = [tool for tool in ("wget", "ffmpeg") if (tool == "wget" or APP.name == "tracking_2d") and not shutil.which(tool)]
if needed:
    if not shutil.which("apt-get") or os.geteuid() != 0:
        raise RuntimeError(f"Faltan herramientas del sistema: {', '.join(needed)}. Utiliza un entorno de Colab.")
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", *needed], check=True)
subprocess.run([sys.executable, "-c", "import torch; print('Dispositivo:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"], check=True)
if APP.name == "radar_trafico":
    subprocess.run([sys.executable, str(APP / "app.py"), "--check"], check=True)
INSTALLED_APP = APP
print("Listo. Ejecuta la siguiente celda.")

## Abrir la aplicación

La siguiente celda permanece activa. Usa el enlace que aparezca en la salida.

Para finalizar, descarga los resultados desde la interfaz y pulsa **Detener** en Colab. El cierre espera hasta 15 segundos antes de forzar la salida; un análisis interrumpido puede quedar incompleto.

In [ ]:
# @title 4. Ejecutar la experiencia
def run_app(script, env):
    # Cada ejecución posee su grupo para cerrar también túnel y procesos hijos.
    with socket.socket() as probe:
        probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            probe.bind(("127.0.0.1", 8000))
        except OSError as exc:
            raise RuntimeError("El puerto 8000 está ocupado. Detén la ejecución anterior antes de continuar.") from exc
    process = subprocess.Popen(
        [sys.executable, "-u", str(script)], cwd=ROOT, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        bufsize=1, start_new_session=True,
    )
    interrupted = False
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)
    except KeyboardInterrupt:
        interrupted = True
        print("\nCerrando la experiencia…", flush=True)
    finally:
        try:
            os.killpg(process.pid, signal.SIGINT)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            print("Se agotó el tiempo de cierre; se forzará la salida.", flush=True)
        finally:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
            process.stdout.close()
    if interrupted:
        print("Experiencia detenida. Ya puedes cambiar la selección.")

if APP is None or INSTALLED_APP != APP:
    raise RuntimeError("Instala los requisitos de la experiencia seleccionada antes de ejecutarla.")
env = os.environ.copy()
env["NPOOL"] = "1"
env["JOBS_DIR"] = str(ROOT / ("traffic-monitor" if APP.name == "radar_trafico" else "jobs"))
run_app(APP / "app.py", env)

## Guardar resultados

- **Tracking en video:** `video.mp4`, `mapa_calor.png`, `trayectorias.csv` y `resultados.json`.
- **Tracking en webcam:** `trayectorias_en_vivo.csv` al pulsar «Detener y guardar».
- **Cámara móvil planar (etapa 02.d):** `comparacion_planar.csv` al detener.
- **Radar:** video anotado, `vehicles.csv`, `heatmap.png` y `traffic-by-time.png`.
- **Odometría 3D (etapa 03, nivel 3):** `trayectoria_camara.csv` y `vehiculos.csv` al pulsar «Detener y guardar».

Para el informe, registra video, configuración, calibración y resultados observados. Las velocidades son estimaciones académicas; valida su interpretación con la escena utilizada.